# 06 - Construcción del dataset unificado

Integra en una única tabla, con granularidad diaria y por embalse, la variable objetivo y las variables predictoras procedentes de las distintas fuentes.

## Estructura de salida

Formato largo: una fila por combinación de fecha y embalse, con las variables en columnas. Los bloques de variables son:

- **Objetivo**: porcentaje de llenado.
- **Hidrología del embalse**: caudal de aportación y de salida.
- **Meteorología**: agregado de las estaciones asignadas por proximidad (SAIH y AEMET).
- **Calidad del agua**: agregado de las estaciones asignadas por topología, con distinción entre las situadas aguas arriba y aguas abajo.

Cuando un embalse tiene varias estaciones asignadas para un mismo tipo de variable, los valores se agregan mediante la media aritmética de las estaciones disponibles cada día.

## Salida

- `data/processed/dataset_unificado.parquet`

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.append("..")
from src.saih import PATRON_COL, normalizar_id_embalse, id_estacion

DIR_RAW = Path("../data/raw")
DIR_INTERIM = Path("../data/interim")
DIR_PROCESSED = Path("../data/processed")

FECHA_INICIO = pd.Timestamp("2006-01-01")
FECHA_FIN = pd.Timestamp("2023-12-31")

TRAIN_INI, TRAIN_FIN = FECHA_INICIO, pd.Timestamp("2021-12-31")
TEST_INI, TEST_FIN = pd.Timestamp("2022-01-01"), FECHA_FIN

maestro = pd.read_parquet(DIR_PROCESSED / "maestro_embalses.parquet")

# E16D (As Portas - C.H. Conso) es la central hidroeléctrica, no un embalse con
# capacidad de regulación: su serie de llenado es constante en torno al 0%. Se
# excluye del dataset de modelado.
EXCLUIDOS = ["E16D"]
ids_validos = set(maestro["ID_SAIH"]) - set(EXCLUIDOS)

asig_meteo = pd.read_parquet(DIR_PROCESSED / "asignacion_meteo_embalses.parquet")
asig_calidad = pd.read_parquet(DIR_PROCESSED / "asignacion_calidad_embalses.parquet")

print(f"Embalses: {len(maestro)}")
print(f"Pares embalse-estación meteorológica: {len(asig_meteo)}")
print(f"Pares embalse-estación de calidad: {len(asig_calidad)}")

Embalses: 35
Pares embalse-estación meteorológica: 367
Pares embalse-estación de calidad: 30


### Periodo de estudio

El periodo se acota a 2006-2023. Los registros de caudal de aportación, variable predictora central del escenario base, están ausentes con anterioridad a 2006 en once embalses, lo que dejaría a la muestra sin homogeneidad en el primer tercio de la serie. Con el recorte, la disponibilidad de esta variable en el conjunto experimental pasa del
91,1% al 99,6%.

El periodo resultante comprende 18 años, de los cuales los dos últimos (2022-2023) se reservan como conjunto de test. Como contrapartida, queda fuera del análisis el episodio de sequía de 2004-2005, circunstancia que se recoge en el capítulo de limitaciones.

## 1. Utilidades de carga

Las hojas de datos del SAIH presentan las series en formato ancho, con una columna por combinación de estación y variable. Se define una función que las convierte a formato largo, resolviendo el identificador de estación y el código de variable a partir del nombre de columna.

In [2]:
def cargar_hoja(fichero, hoja, normalizar_embalses=False):
    """Carga una hoja del SAIH en formato largo: fecha, estacion, variable, valor."""
    df = pd.read_excel(DIR_RAW / "saih" / fichero, sheet_name=hoja, skiprows=[1, 2])
    df = df.rename(columns={df.columns[0]: "fecha"})
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    mapa = {}
    for c in df.columns[1:]:
        m = PATRON_COL.match(str(c).strip())
        if m:
            est, var = m.group(1).upper(), m.group(2).upper()
            if normalizar_embalses and est not in ids_validos:
                est = normalizar_id_embalse(est)
            if est:
                mapa[c] = (est, var)

    largo = (df[["fecha"] + list(mapa)]
             .melt(id_vars="fecha", var_name="col", value_name="valor")
             .dropna(subset=["valor"]))
    largo["estacion"] = largo["col"].map(lambda c: mapa[c][0])
    largo["variable"] = largo["col"].map(lambda c: mapa[c][1])

    return largo.drop(columns="col").query("@FECHA_INICIO <= fecha <= @FECHA_FIN")

## 2. Variable objetivo y variables hidrológicas del embalse

El porcentaje de llenado se obtiene del volumen embalsado y la capacidad máxima del maestro de embalses. Se incorporan además el caudal de aportación y el de salida, ambos registrados en el propio embalse y disponibles para la totalidad de los mismos.

In [3]:
VOLUMEN = {"MACVEMBA", "MAIVEMBA", "ACVEMBA"}
APORTACION = {"MAIAPORT", "ACAPORT"}
SALIDA = {"MACQSALR", "MAIQSALR", "MAIQTSAL", "MACQTSAL", "ACQTSAL"}

emb_largo = cargar_hoja("Datos_Embalses.xlsx", "Datos_Embalses", normalizar_embalses=True)
emb_largo = emb_largo[emb_largo["estacion"].isin(ids_validos)]

def bloque(codigos, nombre):
    return (emb_largo[emb_largo["variable"].isin(codigos)]
            .groupby(["fecha", "estacion"], as_index=False)["valor"].mean()
            .rename(columns={"estacion": "ID_SAIH", "valor": nombre}))

volumen = bloque(VOLUMEN, "volumen_hm3")
aportacion = bloque(APORTACION, "aportacion_m3s")
salida = bloque(SALIDA, "salida_m3s")

base = (volumen
        .merge(maestro[["ID_SAIH", "Capacidad_hm3"]], on="ID_SAIH", how="left")
        .assign(pct_llenado=lambda d: d["volumen_hm3"] / d["Capacidad_hm3"] * 100)
        .merge(aportacion, on=["fecha", "ID_SAIH"], how="left")
        .merge(salida, on=["fecha", "ID_SAIH"], how="left")
        .drop(columns="Capacidad_hm3"))

print(f"Registros: {len(base):,} | embalses: {base['ID_SAIH'].nunique()}")
print(f"Rango: {base['fecha'].min():%Y-%m-%d} a {base['fecha'].max():%Y-%m-%d}\n")
print("Cobertura de las variables del embalse:")
for c in ["volumen_hm3", "pct_llenado", "aportacion_m3s", "salida_m3s"]:
    print(f"  {c:<18} {base[c].notna().mean()*100:5.1f}% de los registros")

Registros: 216,127 | embalses: 34
Rango: 2006-01-01 a 2023-12-31

Cobertura de las variables del embalse:
  volumen_hm3        100.0% de los registros
  pct_llenado        100.0% de los registros
  aportacion_m3s      99.8% de los registros
  salida_m3s          99.5% de los registros


## 3. Variables meteorológicas

Se agregan las estaciones asignadas a cada embalse por proximidad, combinando las observaciones del SAIH (precipitación y temperatura ambiente) con las de AEMET, que aportan además humedad, viento, insolación y presión.

Para cada embalse y día se calcula la media de las estaciones con dato disponible.

In [4]:
# --- SAIH: precipitación y temperatura ---
prec = cargar_hoja("Datos_Estaciones.xlsx", "Datos_Precipitación")
temp = cargar_hoja("Datos_Estaciones.xlsx", "Datos_TempAmb")
meteo_saih = pd.concat([prec, temp], ignore_index=True)

VAR_SAIH = {"AIPCINC": "precipitacion_mm", "AITEMEX": "temp_media_c"}
meteo_saih["nombre"] = meteo_saih["variable"].map(VAR_SAIH)

saih_emb = (asig_meteo.query("fuente == 'SAIH'")[["ID_emb", "ID_est"]]
            .merge(meteo_saih, left_on="ID_est", right_on="estacion")
            .groupby(["fecha", "ID_emb", "nombre"], as_index=False)["valor"].mean()
            .pivot(index=["fecha", "ID_emb"], columns="nombre", values="valor")
            .reset_index())

print(f"SAIH -> registros: {len(saih_emb):,} | embalses: {saih_emb['ID_emb'].nunique()}")
print(saih_emb.head(3).to_string(index=False))

SAIH -> registros: 194,946 | embalses: 35
     fecha ID_emb  precipitacion_mm  temp_media_c
2008-10-01   E001               0.0     13.516667
2008-10-01   E002               0.0     14.328571
2008-10-01   E003               0.0      8.850000


In [5]:
# --- AEMET ---
aemet = pd.read_parquet(DIR_INTERIM / "aemet_minosil_2000_2023.parquet")
aemet["fecha"] = pd.to_datetime(aemet["fecha"])

VAR_AEMET = {
    "tmed": "aemet_temp_media_c", "tmin": "aemet_temp_min_c", "tmax": "aemet_temp_max_c",
    "prec": "aemet_precipitacion_mm", "hrMedia": "aemet_humedad_pct",
    "velmedia": "aemet_viento_ms", "sol": "aemet_insolacion_h",
    "presMax": "aemet_presion_max_hpa", "presMin": "aemet_presion_min_hpa",
}

# Las variables numéricas de AEMET usan coma decimal
disponibles = [c for c in VAR_AEMET if c in aemet.columns]
for c in disponibles:
    aemet[c] = pd.to_numeric(
        aemet[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

aemet_emb = (asig_meteo.query("fuente == 'AEMET'")[["ID_emb", "ID_est"]]
             .merge(aemet[["fecha", "indicativo"] + disponibles],
                    left_on="ID_est", right_on="indicativo")
             .groupby(["fecha", "ID_emb"], as_index=False)[disponibles].mean()
             .rename(columns=VAR_AEMET))

print(f"AEMET -> registros: {len(aemet_emb):,} | embalses: {aemet_emb['ID_emb'].nunique()}")
print(f"Variables: {[VAR_AEMET[c] for c in disponibles]}")
print(f"\nCobertura por variable:")
for c in disponibles:
    print(f"  {VAR_AEMET[c]:<26} {aemet_emb[VAR_AEMET[c]].notna().mean()*100:5.1f}%")

AEMET -> registros: 289,730 | embalses: 35
Variables: ['aemet_temp_media_c', 'aemet_temp_min_c', 'aemet_temp_max_c', 'aemet_precipitacion_mm', 'aemet_humedad_pct', 'aemet_viento_ms', 'aemet_insolacion_h', 'aemet_presion_max_hpa', 'aemet_presion_min_hpa']

Cobertura por variable:
  aemet_temp_media_c          98.4%
  aemet_temp_min_c            98.4%
  aemet_temp_max_c            98.4%
  aemet_precipitacion_mm      98.0%
  aemet_humedad_pct           92.4%
  aemet_viento_ms             80.6%
  aemet_insolacion_h          50.4%
  aemet_presion_max_hpa       65.2%
  aemet_presion_min_hpa       65.2%


In [6]:
sin_aemet = set(asig_meteo["ID_emb"]) - set(aemet_emb["ID_emb"])
print(f"Embalses sin estación AEMET asignada: {sorted(sin_aemet)}")

Embalses sin estación AEMET asignada: []


### Cobertura temporal de las estaciones meteorológicas del SAIH

Se comprueba la disponibilidad temporal de las series meteorológicas del SAIH frente a las de AEMET, dado que ambas fuentes aportan precipitación y temperatura y su
solapamiento condiciona cuál emplear como fuente principal.

In [7]:
saih_anual = (meteo_saih[meteo_saih["variable"] == "AIPCINC"]
              .assign(anio=lambda d: d["fecha"].dt.year)
              .groupby("anio")["estacion"].nunique())

aemet_anual = (aemet.assign(anio=lambda d: d["fecha"].dt.year)
               .groupby("anio")["prec"].apply(lambda s: s.notna().sum()))

print("Estaciones del SAIH con registros de precipitación por año:")
print(f"  {saih_anual.to_dict()}\n")
print("Registros de precipitación de AEMET por año:")
print(f"  {aemet_anual.to_dict()}")

Estaciones del SAIH con registros de precipitación por año:
  {2008: 72, 2009: 74, 2010: 74, 2011: 87, 2012: 88, 2013: 88, 2014: 88, 2015: 87, 2016: 87, 2017: 87, 2018: 87, 2019: 87, 2020: 89, 2021: 91, 2022: 91, 2023: 91}

Registros de precipitación de AEMET por año:
  {2000: 3813, 2001: 4587, 2002: 4964, 2003: 4686, 2004: 5153, 2005: 5560, 2006: 5812, 2007: 5681, 2008: 5215, 2009: 6384, 2010: 6564, 2011: 7122, 2012: 7167, 2013: 7477, 2014: 7609, 2015: 7926, 2016: 7610, 2017: 7773, 2018: 7637, 2019: 7550, 2020: 7849, 2021: 7624, 2022: 7938, 2023: 7909}


Las series meteorológicas del SAIH no presentan registros con anterioridad a 2008 y alcanzan cobertura completa únicamente a partir de 2009, mientras que las de AEMET cubren la totalidad del periodo de estudio con una disponibilidad situada entre el 75% y el 94% según el año.

Se adopta por tanto AEMET como fuente meteorológica del trabajo. Emplear ambas de forma combinada, utilizando el SAIH para completar los registros ausentes a partir de 2009, elevaría la cobertura en la segunda mitad del periodo, pero introduciría una inhomogeneidad en la naturaleza de la serie que los modelos podrían interpretar como señal. Las series meteorológicas del SAIH se descartan por este motivo, quedando documentada su disponibilidad parcial.

## 4. Variables de calidad del agua

### Cobertura temporal de las estaciones de calidad

Antes de determinar qué embalses admiten el escenario con variables de calidad, se comprueba la disponibilidad temporal de cada estación. Once de las dieciocho estaciones disponen de serie desde 2001, mientras que cinco entraron en servicio en 2020 y dos en 2011. Ninguna registra datos en el año 2000, lo cual no es un problema ya que hemos recortado el periodo de estudio a partir de 2006.

In [8]:
calidad_raw = pd.read_excel(DIR_RAW / "saih" / "Datos_Estaciones_Calidad.xlsx",
                            sheet_name="Datos_Calidad", skiprows=[1, 2])
calidad_raw = calidad_raw.rename(columns={calidad_raw.columns[0]: "fecha"})
calidad_raw["fecha"] = pd.to_datetime(calidad_raw["fecha"])

filas = []
for col in calidad_raw.columns[1:]:
    est = id_estacion(col)
    s = calidad_raw.loc[calidad_raw[col].notna(), "fecha"]
    if est and len(s):
        filas.append({
            "ID": est,
            "variable": PATRON_COL.match(str(col).strip()).group(2).upper(),
            "n_dias": len(s),
            "desde": s.min(),
            "hasta": s.max(),
        })

cob = pd.DataFrame(filas)

resumen_cob = (cob.groupby("ID")
               .agg(n_variables=("variable", "nunique"),
                    dias_max=("n_dias", "max"),
                    desde=("desde", "min"),
                    hasta=("hasta", "max"))
               .reset_index())
resumen_cob["anios"] = ((resumen_cob["hasta"] - resumen_cob["desde"]).dt.days / 365.25).round(1)

resumen_cob["asignada"] = resumen_cob["ID"].isin(set(asig_calidad["ID_cal"]))

print(resumen_cob.sort_values("dias_max", ascending=False).to_string(index=False))

  ID  n_variables  dias_max      desde      hasta  anios  asignada
A043            7      8130 2001-01-01 2024-05-20   23.4     False
N001            7      8026 2001-01-01 2024-05-20   23.4     False
A015            6      7913 2001-01-01 2024-05-20   23.4     False
A033            7      7834 2001-01-01 2024-05-20   23.4      True
A008            6      7538 2001-01-03 2024-05-20   23.4     False
N013            6      7393 2001-01-01 2024-05-20   23.4      True
N007            6      7336 2001-07-12 2024-05-20   22.9      True
A046            8      7312 2001-01-01 2024-05-20   23.4      True
N010            7      7307 2001-07-31 2024-05-20   22.8      True
N015            6      7191 2001-04-26 2024-05-20   23.1      True
Q123            6      4906 2001-01-01 2024-05-07   23.3      True
A041            5      4889 2010-12-17 2024-05-20   13.4      True
A022            5      4797 2010-12-16 2024-05-20   13.4     False
A044            6      1548 2020-02-19 2024-05-20    4.2     F

Se agregan las estaciones de calidad asignadas a cada embalse, distinguiendo según su posición relativa: las situadas aguas arriba miden agua que aún no ha alcanzado el embalse, mientras que las situadas aguas abajo miden agua ya regulada. Al responder a mecanismos distintos, ambos grupos se mantienen separados para permitir el análisis diferenciado de su aportación.

In [9]:
VAR_CALIDAD = {
    "AIA3ATS": "amonio_mgl", "AIA6AFS": "fosfatos_mgl", "AIMPCTS": "conductividad_uscm",
    "AIMPO2S": "oxigeno_mgl", "AIMPPHS": "ph", "AIMPTTS": "temp_agua_c",
    "AITUTUS": "turbidez_ntu", "AIMOMOS": "materia_organica_m1",
}

calidad = cargar_hoja("Datos_Estaciones_Calidad.xlsx", "Datos_Calidad")
calidad["nombre"] = calidad["variable"].map(VAR_CALIDAD)

bloques = []
for direccion in ["arriba", "abajo"]:
    pares = asig_calidad.query("relacion == @direccion")[["ID_emb", "ID_cal"]]
    b = (pares.merge(calidad, left_on="ID_cal", right_on="estacion")
         .groupby(["fecha", "ID_emb", "nombre"], as_index=False)["valor"].mean()
         .pivot(index=["fecha", "ID_emb"], columns="nombre", values="valor")
         .reset_index())
    b.columns = [c if c in ("fecha", "ID_emb") else f"cal_{direccion}_{c}" for c in b.columns]
    bloques.append(b)
    print(f"{direccion}: {len(b):,} registros | {b['ID_emb'].nunique()} embalses")

calidad_emb = bloques[0].merge(bloques[1], on=["fecha", "ID_emb"], how="outer")
print(f"\nTotal: {len(calidad_emb):,} registros | {calidad_emb['ID_emb'].nunique()} embalses")
print(f"\nCobertura por variable:")
for c in sorted(calidad_emb.columns):
    if c.startswith("cal_"):
        print(f"  {c:<34} {calidad_emb[c].notna().mean()*100:5.1f}%")

arriba: 55,779 registros | 11 embalses
abajo: 84,105 registros | 15 embalses

Total: 120,908 registros | 22 embalses

Cobertura por variable:
  cal_abajo_amonio_mgl                60.2%
  cal_abajo_conductividad_uscm        69.3%
  cal_abajo_materia_organica_m1       27.8%
  cal_abajo_oxigeno_mgl               68.2%
  cal_abajo_ph                        69.0%
  cal_abajo_temp_agua_c               69.3%
  cal_abajo_turbidez_ntu              66.9%
  cal_arriba_amonio_mgl               40.2%
  cal_arriba_conductividad_uscm       45.4%
  cal_arriba_fosfatos_mgl              4.3%
  cal_arriba_materia_organica_m1       8.1%
  cal_arriba_oxigeno_mgl              45.0%
  cal_arriba_ph                       45.5%
  cal_arriba_temp_agua_c              45.6%
  cal_arriba_turbidez_ntu             44.4%


In [10]:
rejilla = pd.MultiIndex.from_product(
    [pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D"),
     sorted(asig_calidad["ID_emb"].unique())],
    names=["fecha", "ID_emb"]).to_frame(index=False)

chequeo = rejilla.merge(calidad_emb, on=["fecha", "ID_emb"], how="left")

cols = ["cal_arriba_ph", "cal_arriba_conductividad_uscm",
        "cal_abajo_ph", "cal_abajo_conductividad_uscm"]
print((chequeo.groupby("ID_emb")[cols].apply(lambda g: g.notna().mean() * 100)
       .round(1).to_string()))

        cal_arriba_ph  cal_arriba_conductividad_uscm  cal_abajo_ph  cal_abajo_conductividad_uscm
ID_emb                                                                                          
E001             21.0                           21.0           0.0                           0.0
E002              0.0                            0.0          91.9                          92.0
E008              0.0                            0.0          92.5                          92.7
E009              0.0                            0.0          92.5                          92.7
E011             92.5                           92.7          55.7                          56.1
E013             55.7                           56.1           0.0                           0.0
E023             55.7                           56.1           0.0                           0.0
E024             55.7                           56.1           0.0                           0.0
E025              0.0         

In [11]:
for emb in ["E013", "E024", "E350", "E001"]:
    s = chequeo[chequeo["ID_emb"] == emb].set_index("fecha")["cal_arriba_ph"].fillna(
        chequeo[chequeo["ID_emb"] == emb].set_index("fecha")["cal_abajo_ph"])
    anual = s.notna().groupby(s.index.year).mean() * 100
    print(f"{emb}: {anual.round(0).astype(int).to_dict()}")

E013: {2006: 57, 2007: 80, 2008: 98, 2009: 75, 2010: 92, 2011: 67, 2012: 60, 2013: 72, 2014: 70, 2015: 52, 2016: 55, 2017: 10, 2018: 67, 2019: 36, 2020: 45, 2021: 26, 2022: 9, 2023: 30}
E024: {2006: 57, 2007: 80, 2008: 98, 2009: 75, 2010: 92, 2011: 67, 2012: 60, 2013: 72, 2014: 70, 2015: 52, 2016: 55, 2017: 10, 2018: 67, 2019: 36, 2020: 45, 2021: 26, 2022: 9, 2023: 30}
E350: {2006: 57, 2007: 80, 2008: 98, 2009: 75, 2010: 92, 2011: 67, 2012: 60, 2013: 72, 2014: 70, 2015: 52, 2016: 55, 2017: 10, 2018: 67, 2019: 36, 2020: 45, 2021: 26, 2022: 9, 2023: 30}
E001: {2006: 0, 2007: 0, 2008: 0, 2009: 0, 2010: 0, 2011: 0, 2012: 0, 2013: 0, 2014: 0, 2015: 0, 2016: 0, 2017: 0, 2018: 0, 2019: 0, 2020: 79, 2021: 100, 2022: 100, 2023: 100}


In [12]:
# Check cobertura test
cal_cols = [c for c in chequeo.columns if c.startswith("cal_")]
chequeo["tiene_calidad"] = chequeo[cal_cols].notna().any(axis=1)

periodos = {
    "train_2006_2021": (TRAIN_INI, TRAIN_FIN),
    "test_2022_2023": (TEST_INI, TEST_FIN),
}

res = []
for emb, g in chequeo.groupby("ID_emb"):
    fila = {"ID_emb": emb}
    for nombre, (ini, fin) in periodos.items():
        sub = g[(g["fecha"] >= ini) & (g["fecha"] <= fin)]
        fila[nombre] = round(sub["tiene_calidad"].mean() * 100, 1)
    res.append(fila)

res = pd.DataFrame(res).sort_values("test_2022_2023")
print(res.to_string(index=False))

ID_emb  train_2006_2021  test_2022_2023
  E023             61.3            19.3
  E024             61.3            19.3
  E013             61.3            19.3
  E350             61.3            19.3
  E35A             92.5            96.0
  E028             96.1            98.9
  E027             91.2            99.5
  E026             91.2            99.5
  E030             91.2            99.5
  E031             91.2            99.5
  E025             91.2            99.5
  E002             91.2            99.5
  E011             95.0           100.0
  E001             11.2           100.0
  E008             92.0           100.0
  E009             92.0           100.0
  E033             97.6           100.0
  E029             97.3           100.0
  E32A             87.2           100.0
  E07A             92.0           100.0
  E570             95.0           100.0
  E571             95.0           100.0


In [13]:
MIN_COBERTURA = 50

experimentales = sorted(res.query(
    "train_2006_2021 >= @MIN_COBERTURA and test_2022_2023 >= @MIN_COBERTURA")["ID_emb"])

excluidos = sorted(set(res["ID_emb"]) - set(experimentales))
print(f"Conjunto experimental: {len(experimentales)} embalses")
print(experimentales)
print(f"\nExcluidos por cobertura insuficiente: {excluidos}")

pd.DataFrame({"ID_SAIH": experimentales}).to_parquet(
    DIR_PROCESSED / "embalses_experimento_calidad.parquet", index=False)

Conjunto experimental: 17 embalses
['E002', 'E008', 'E009', 'E011', 'E025', 'E026', 'E027', 'E028', 'E029', 'E030', 'E031', 'E033', 'E07A', 'E32A', 'E35A', 'E570', 'E571']

Excluidos por cobertura insuficiente: ['E001', 'E013', 'E023', 'E024', 'E350']


### Conjunto experimental para el escenario con variables de calidad

De los 22 embalses con estación de calidad asignada, cinco presentan cobertura temporal insuficiente para el diseño experimental:

- E013, E023, E024 y E350 dependen de la estación Q123 (Sil en O Barco de Valdeorras),   cuya serie se interrumpe en gran medida a partir de 2021, dejando el periodo reservado para test con una cobertura del 19%.
- E001 (Belesar) depende de N002 (Sarria), estación puesta en servicio en 2020, lo que reduce su cobertura en el periodo de entrenamiento al 8,5%.

Se establece como criterio de admisión una cobertura mínima del 50% tanto en el periodo de entrenamiento como en el de test, de modo que las métricas de evaluación no queden determinadas por el procedimiento de imputación. El conjunto experimental resultante comprende 17 embalses.

Cabe señalar que el mayor embalse de la cuenca queda fuera del análisis con variables de calidad, lo que ilustra la limitación de cobertura de la red de control descrita en el apartado anterior.

## 5. Integración

Se construye la rejilla completa de fechas y embalses del periodo de estudio y se incorporan los distintos bloques de variables. El formato resultante mantiene los valores ausentes explícitos, cuyo tratamiento se aborda en el análisis de completitud.

Quedan excluidos del dataset:

- Las series meteorológicas del SAIH, por no cubrir el periodo anterior a 2008.
- Los fosfatos y la materia orgánica, con una disponibilidad del 3,5% y el 6,7% respectivamente en las estaciones situadas aguas arriba, insuficiente para su incorporación al modelado.

Se mantienen en cambio la insolación y la presión atmosférica de AEMET pese a su menor cobertura, al no comprometer la homogeneidad temporal de la serie. Su inclusión efectiva en los modelos se decide en la fase de ingeniería de variables.

In [14]:
DESCARTADAS = ["fosfatos_mgl", "materia_organica_m1"]

rejilla = pd.MultiIndex.from_product(
    [pd.date_range(FECHA_INICIO, FECHA_FIN, freq="D"), sorted(ids_validos)],
    names=["fecha", "ID_SAIH"]).to_frame(index=False)

calidad_final = calidad_emb.drop(
    columns=[c for c in calidad_emb.columns if any(d in c for d in DESCARTADAS)])

dataset = (rejilla
           .merge(base, on=["fecha", "ID_SAIH"], how="left")
           .merge(aemet_emb.rename(columns={"ID_emb": "ID_SAIH"}),
                  on=["fecha", "ID_SAIH"], how="left")
           .merge(calidad_final.rename(columns={"ID_emb": "ID_SAIH"}),
                  on=["fecha", "ID_SAIH"], how="left")
           .merge(maestro[["ID_SAIH", "Nombre_SAIH", "Sistema", "Capacidad_hm3"]],
                  on="ID_SAIH", how="left"))

dataset.to_parquet(DIR_PROCESSED / "dataset_unificado.parquet", index=False)

print(f"Dimensiones: {dataset.shape[0]:,} filas x {dataset.shape[1]} columnas")
print(f"Periodo: {dataset['fecha'].min():%Y-%m-%d} a {dataset['fecha'].max():%Y-%m-%d}\n")
print("Cobertura por variable:")
for c in dataset.columns:
    if c not in ("fecha", "ID_SAIH", "Nombre_SAIH", "Sistema", "Capacidad_hm3"):
        print(f"  {c:<32} {dataset[c].notna().mean()*100:5.1f}%")

Dimensiones: 223,516 filas x 30 columnas
Periodo: 2006-01-01 a 2023-12-31

Cobertura por variable:
  volumen_hm3                       96.7%
  pct_llenado                       96.7%
  aportacion_m3s                    96.5%
  salida_m3s                        96.2%
  aemet_temp_media_c                96.9%
  aemet_temp_min_c                  96.9%
  aemet_temp_max_c                  96.9%
  aemet_precipitacion_mm            96.0%
  aemet_humedad_pct                 94.1%
  aemet_viento_ms                   85.5%
  aemet_insolacion_h                57.5%
  aemet_presion_max_hpa             64.1%
  aemet_presion_min_hpa             64.1%
  cal_arriba_amonio_mgl             21.8%
  cal_arriba_conductividad_uscm     24.6%
  cal_arriba_oxigeno_mgl            24.3%
  cal_arriba_ph                     24.6%
  cal_arriba_temp_agua_c            24.7%
  cal_arriba_turbidez_ntu           24.0%
  cal_abajo_amonio_mgl              32.6%
  cal_abajo_conductividad_uscm      37.5%
  cal_abajo_oxigeno

In [15]:
# En el total, los % de completitud de las variables de calidad salen muy bajo por estudiarlos en el total de embalses y no tener en cuenta la existencia o no de estaciones de calidad aguas arriba o abajo
# Se hace revisión específica de % de completitud por embalse (solo los que entrarán en el estudio) y bloque (arriba/abajo)
exp = pd.read_parquet(DIR_PROCESSED / "embalses_experimento_calidad.parquet")["ID_SAIH"]
sub = dataset[dataset["ID_SAIH"].isin(exp) & (dataset["fecha"] >= FECHA_INICIO)]

arriba = [c for c in sub.columns if c.startswith("cal_arriba_")]
abajo = [c for c in sub.columns if c.startswith("cal_abajo_")]

cob_bloque = pd.DataFrame({
    "arriba_%": sub.groupby("ID_SAIH")[arriba].apply(lambda g: g.notna().any(axis=1).mean() * 100),
    "abajo_%": sub.groupby("ID_SAIH")[abajo].apply(lambda g: g.notna().any(axis=1).mean() * 100),
}).round(1)
cob_bloque["alguna_%"] = (sub.groupby("ID_SAIH")[arriba + abajo]
                          .apply(lambda g: g.notna().any(axis=1).mean() * 100).round(1))
print(cob_bloque.to_string())

         arriba_%  abajo_%  alguna_%
ID_SAIH                             
E002          0.0     92.1      92.1
E008          0.0     92.9      92.9
E009          0.0     92.9      92.9
E011         92.9     56.7      95.6
E025          0.0     92.1      92.1
E026          0.0     92.1      92.1
E027          0.0     92.1      92.1
E028          0.0     96.5      96.5
E029         96.5     92.1      97.6
E030          0.0     92.1      92.1
E031         92.1      0.0      92.1
E033         94.6     93.0      97.9
E07A          0.0     92.9      92.9
E32A          0.0     88.6      88.6
E35A         87.1      0.0      87.1
E570         95.6      0.0      95.6
E571         92.9     56.7      95.6


In [16]:
# Está incluido Belesar?
exp = pd.read_parquet(DIR_PROCESSED / "embalses_experimento_calidad.parquet")["ID_SAIH"]
print(f"Experimentales: {len(exp)}")
print(f"E001 en experimentales: {'E001' in set(exp)}")

Experimentales: 17
E001 en experimentales: False
